In [1]:
# %% [markdown]
# # 04b — Sparse MLP Experiment (3-Seed Reproducibility)
#
# Same structural sparsity as Sparse KAN (notebook 03b) but with
# nn.Linear + SiLU instead of KANLinear B-splines.
#
# Run AFTER 04a_sparse_mlp_validation.py confirms all tests pass.
#
# Architecture counts are read directly from the live model at runtime
# (taxonomy may have changed — no hardcoded numbers). This was already
# true before this edit pass: SparseMLP.from_taxonomy() derives
# n_subthemes/n_themes independently per dataset from whichever taxonomy_df
# is passed in, so agg_means and agg_full_moments naturally get different
# widths without any code change here. (Compare to the Dense MLP notebook,
# which previously hardcoded a single shared N_SUBTHEMES/N_THEMES pair and
# needed fixing for exactly this reason.)
#
# EDIT (post Stage_5_Themes/04_numbering_subthemes.ipynb):
#   - "feature_set" renamed to "dataset" throughout, values renamed to
#     "agg_full_moments"/"agg_means" -- matches data_utils.py's contract
#     and the actual file stems written by 05_splits.ipynb.
#   - Taxonomy loaded via data_utils.load_theme_assignment() (dtype=str
#     enforced on ID columns) rather than a raw pd.read_csv, and pointed
#     at the new numbered taxonomy files.
#   - Continuous target is now minret_5d_pct (raw percentage), not
#     minret_5d_z (expanding z-score) -- see data_utils.py docstring.
#   - HuberLoss delta is now loaded per-split from huber_delta.json rather
#     than the fixed module constant delta=3, and threaded through both
#     Optuna phases and the final retrain.
#   - RESULTS_DIR moved to a fresh folder -- Optuna study names carry no
#     data-version marker (load_if_exists=True keyed only on
#     model/target/split/phase/seed), so reusing the old folder would
#     silently resume trials whose objective values were computed on the
#     old feature set / old target scale.
#   - Backtest results are now SAVED to disk (backtest_summary.csv +
#     backtest_full_results.json), not just printed.
#
# 3 seeds × 4 splits × 2 datasets × 2 targets = 48 runs.
# Each seed runs the FULL pipeline independently:
#   - Optuna Phase 1: N_TRIALS_NO_L1 trials, no L1  (seeded TPE sampler)
#   - Optuna Phase 2: N_TRIALS trials, with L1       (seeded TPE sampler)
#   - Best across both phases used for final retrain
#   - Evaluation on train/val/test
#
# This measures true end-to-end reproducibility: if Optuna picks
# different hyperparameters across seeds, that tells us the search
# landscape is flat or noisy. If metrics vary, that tells us the
# model is sensitive to initialisation. Both are important findings.
#
# Target notes:
#   binary:     y_binary (minret_5d_pct < -2.0), BCEWithLogitsLoss, early stop on AUC
#   continuous: minret_5d_pct (raw percentage, NOT a z-score),
#               HuberLoss(delta=<per-split value from huber_delta.json>),
#               early stop on R²
#   Both use Optuna direction="maximize" (AUC and R² are both higher-is-better).
#
# Results save to Drive per-seed as they go — safe against disconnection.
# Aggregated cross-seed summary saved at the end. Backtest results also
# saved to disk this run.
#
# Estimated runtime on T4: ~3 seeds × estimated per-seed time (see timing cell)
# Runtime disconnects automatically when finished.

# %%
# ── COLAB SETUP ──
!pip install -q optuna

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import numpy as np
import pandas as pd
import random
import time
import torch
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import (
    evaluate_model, save_predictions, compute_calibration,
    load_predictions, run_full_backtest,
)
from sparse_mlp import SparseMLP, sparse_mlp_weight_l1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

# EDIT: fresh results dir -- see EDIT block in the markdown header above.
RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_mlp")

# EDIT: "feature_set" -> "dataset", values renamed to match data_utils.py.
DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

# ── Reproducibility seeds ──
# Each seed runs the full pipeline independently (Optuna + retrain + eval).
# Results are reported as mean ± std across seeds.
SEEDS = [42, 123, 456]

# ── Optuna settings ──
N_TRIALS       = 40   # with L1
N_TRIALS_NO_L1 = 30   # without L1

# ── Load Huber deltas once, keyed off split name and target family ──
# 'market' is the aggregate-side key in huber_delta.json (04_targets.ipynb
# computes deltas for 'market' and 'panel' separately). This notebook only
# ever touches agg_means/agg_full_moments, so always 'market'.
with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]


# ═══════════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    """Set all random seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD TAXONOMIES AND SANITY CHECK
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("Loading taxonomies...")
taxonomy_dfs = {}
for ds in DATASETS:
    df = load_theme_assignment(ds, THEMES_DIR)
    taxonomy_dfs[ds] = df
    print(f"  {ds}: {len(df)} features, "
          f"{df['subtheme_id'].nunique()} subthemes, "
          f"{df['theme_id'].nunique()} themes")

# ── Quick build check on GPU ──
# Also captures active edge / parameter counts from the actual model,
# so we never need to hardcode them. n_subthemes/n_themes are derived
# independently per dataset from whichever taxonomy_df is passed in --
# no shared width constant exists anywhere in this file.
device = get_device()
architecture_info = {}

for ds in DATASETS:
    data        = load_split("Split_A", ds, SPLITS_DIR)
    model_check = SparseMLP.from_taxonomy(
        taxonomy_dfs[ds], data["feature_cols"]
    ).to(device)

    x   = torch.randn(4, data["n_features"]).to(device)
    out = model_check(x)
    assert out.shape == (4, 1)
    model_check.verify_masking()

    active_edges  = model_check.count_active_edges()
    active_params = model_check.count_active_parameters()

    architecture_info[ds] = {
        "n_features":        data["n_features"],
        "n_subthemes":       model_check.n_subthemes,
        "n_themes":          model_check.n_themes,
        "active_edges":      active_edges,
        "active_parameters": active_params,
    }

    print(f"  {ds}: {active_edges:,} active edges  "
          f"({active_params:,} active params) — ✓ GPU forward pass OK")
    del model_check, x, out

# ── GPU timing estimate (agg_full_moments — the harder case) ──
data       = load_split("Split_A", "agg_full_moments", SPLITS_DIR)
model_time = SparseMLP.from_taxonomy(
    taxonomy_dfs["agg_full_moments"], data["feature_cols"]
).to(device)
x_bench = torch.randn(256, data["n_features"]).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.time()
for _ in range(500):
    model_time.zero_grad()
    out  = model_time(x_bench)
    loss = out.sum()
    loss.backward()
if device.type == "cuda":
    torch.cuda.synchronize()
fb_ms = (time.time() - t0) / 500 * 1000

n_batches      = max(1, 3000 // 256)
trial_s        = fb_ms / 1000 * n_batches * 50
total_trials   = (N_TRIALS + N_TRIALS_NO_L1) * len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)
est_seed_hours = trial_s * total_trials / 3600

print(f"\n  GPU forward+backward (batch=256): {fb_ms:.2f}ms")
print(f"  Estimated per trial: {trial_s:.1f}s")
print(f"  Estimated per seed ({total_trials} trials): {est_seed_hours:.1f} hours")
print(f"  Estimated total (3 seeds): {est_seed_hours * 3:.1f} hours\n")

del model_time, x_bench, data
if device.type == "cuda":
    torch.cuda.empty_cache()


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL FACTORIES
# ═══════════════════════════════════════════════════════════════════════════════

def make_model_factory_no_l1(feature_cols, taxonomy_df, huber_delta, target_type):
    """3 hyperparameters, no L1. huber_delta is fixed (not tuned), only used
    when target_type == 'continuous'."""
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])

        model = SparseMLP.from_taxonomy(taxonomy_df, feature_cols)

        train_kwargs = {
            "lr":           lr,
            "weight_decay": weight_decay,
            "n_epochs":     300,
            "patience":     20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs

    return factory


def make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type):
    """4 hyperparameters, with L1 [1e-7, 1e-3]. huber_delta is fixed (not
    tuned), only used when target_type == 'continuous'."""
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])
        reg_weight   = trial.suggest_float("reg_weight", 1e-7, 1e-3, log=True)

        model = SparseMLP.from_taxonomy(taxonomy_df, feature_cols)

        train_kwargs = {
            "lr":           lr,
            "weight_decay": weight_decay,
            "reg_fn":       sparse_mlp_weight_l1,
            "reg_weight":   reg_weight,
            "n_epochs":     300,
            "patience":     20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs

    return factory


# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════════

def run_single_experiment(split_name, dataset, target_type, device,
                          seed, seed_results_dir):
    """
    Run two Optuna phases → pick overall best → train → evaluate → save.

    Phase 1: N_TRIALS_NO_L1 trials, NO L1  (seeded TPE sampler)
    Phase 2: N_TRIALS trials, WITH L1      (seeded TPE sampler)

    The best trial across BOTH phases is used for final training.
    Each seed gets its own study DBs and output subfolder so runs
    are fully independent and resumable.

    Optuna direction is always "maximize":
        binary     → maximise val AUC
        continuous → maximise val R²
    """
    model_name = f"sparse_mlp_{dataset}"

    print(f"\n{'─'*60}")
    print(f"  seed={seed} / {dataset} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None

    metric_name = "AUC" if target_type == "binary" else "R²"

    # ── Helper: run one Optuna phase with seeded sampler ──
    def run_optuna_phase(phase_name, factory, n_trials):
        study_name = (f"{model_name}_{target_type}_{split_name}"
                      f"_{phase_name}_seed{seed}")
        study_path = seed_results_dir / "optuna" / f"{study_name}.db"
        study_path.parent.mkdir(parents=True, exist_ok=True)

        study = optuna.create_study(
            study_name=study_name,
            storage=f"sqlite:///{study_path}",
            direction="maximize",
            load_if_exists=True,
            sampler=optuna.samplers.TPESampler(seed=seed),
        )

        def objective(trial):
            model, train_kwargs = factory(trial)
            batch_size = trial.params["batch_size"]

            loaders = get_dataloaders(
                split_name, dataset, SPLITS_DIR,
                target_type=target_type,
                batch_size=batch_size,
            )

            result = train_model(
                model=model,
                train_loader=loaders["train"],
                val_loader=loaders["val"],
                device=device,
                target_type=target_type,
                pos_weight=pos_weight if target_type == "binary" else None,
                verbose=False,
                **train_kwargs,
            )

            return result["best_val_metric"]  # AUC (binary) or R² (continuous)

        existing  = sum(1 for t in study.trials
                        if t.state == optuna.trial.TrialState.COMPLETE)
        remaining = max(0, n_trials - existing)

        if remaining > 0:
            print(f"  {phase_name}: {remaining} trials ({existing} already complete)")
            study.optimize(objective, n_trials=remaining, show_progress_bar=True)
        else:
            print(f"  {phase_name}: {existing} trials already complete — skipping")

        print(f"  {phase_name} best {metric_name}: {study.best_value:.4f}  "
              f"params: {study.best_params}")
        return study

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # ── Phase 1: No L1 ──
    factory_no_l1 = make_model_factory_no_l1(feature_cols, taxonomy_df,
                                             huber_delta, target_type)
    study_no_l1   = run_optuna_phase("no_L1", factory_no_l1, N_TRIALS_NO_L1)

    # ── Phase 2: With L1 ──
    factory_l1 = make_model_factory_with_l1(feature_cols, taxonomy_df,
                                            huber_delta, target_type)
    study_l1   = run_optuna_phase("with_L1", factory_l1, N_TRIALS)

    # ── Pick overall best (both maximise AUC or R², so >= comparison is correct) ──
    no_l1_wins = study_no_l1.best_value >= study_l1.best_value

    if no_l1_wins:
        best_params = study_no_l1.best_params
        best_val    = study_no_l1.best_value
        use_l1      = False
        print(f"\n  → No-L1 wins (seed={seed}, {metric_name}={best_val:.4f})")
    else:
        best_params = study_l1.best_params
        best_val    = study_l1.best_value
        use_l1      = True
        print(f"\n  → With-L1 wins (seed={seed}, {metric_name}={best_val:.4f})")

    # ── FINAL TRAINING with best params ──
    batch_size = best_params.get("batch_size", 128)
    loaders    = get_dataloaders(
        split_name, dataset, SPLITS_DIR,
        target_type=target_type,
        batch_size=batch_size,
    )

    model = SparseMLP.from_taxonomy(taxonomy_df, feature_cols)

    train_kwargs = {
        "lr":           best_params["lr"],
        "weight_decay": best_params["weight_decay"],
        "pos_weight":   pos_weight if target_type == "binary" else None,
        "n_epochs":     300,
        "patience":     20,
    }
    if use_l1:
        train_kwargs["reg_fn"]     = sparse_mlp_weight_l1
        train_kwargs["reg_weight"] = best_params["reg_weight"]
    if target_type == "continuous":
        train_kwargs["huber_delta"] = huber_delta

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        verbose=True,
        log_every=20,
        **train_kwargs,
    )

    # ── EVALUATE on train/val/test ──
    # For continuous: pass y_true_binary explicitly so derive_binary_from_continuous
    # receives correct 0/1 labels (DataLoader carries minret_5d_pct, not binary labels).
    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters=(
                {**best_params, "used_l1": use_l1, "seed": seed,
                 "huber_delta": huber_delta}
                if part == "test" else None
            ),
            results_dir=seed_results_dir,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        all_metrics[part] = metrics

    # ── SAVE CHECKPOINT ──
    # Model config reads counts from the model itself — no hardcoded values.
    ckpt_dir = seed_results_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters={**best_params, "used_l1": use_l1, "seed": seed,
                         "huber_delta": huber_delta},
        model_config={
            "type":              "SparseMLP_Masked",
            "dataset":           dataset,
            "target_type":       target_type,
            "n_features":        data["n_features"],
            "n_subthemes":       model.n_subthemes,
            "n_themes":          model.n_themes,
            "activation":        "SiLU",
            "active_edges":      model.count_active_edges(),
            "active_parameters": model.count_active_parameters(),
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    # ── PRINT SUMMARY ──
    l1_str = (f"reg_weight={best_params['reg_weight']:.2e}"
              if use_l1 else "no L1")
    if target_type == "binary":
        print(f"\n  Results (seed={seed}, {l1_str}):")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {all_metrics['val']['auc']:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
        print(f"    Gap:       "
              f"{all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"\n  Results (seed={seed}, {l1_str}, huber_delta={huber_delta:.4f}):")
        print(f"    Train R²:    {all_metrics['train']['r2']:.4f}  "
              f"(MSE={all_metrics['train']['mse']:.4f})")
        print(f"    Val R²:      {all_metrics['val']['r2']:.4f}  "
              f"(MSE={all_metrics['val']['mse']:.4f})")
        print(f"    Test R²:     {all_metrics['test']['r2']:.4f}  "
              f"(MSE={all_metrics['test']['mse']:.4f})")
        print(f"    Derived AUC: {all_metrics['test']['derived_auc']:.4f}")
        print(f"    Pred std:    {all_metrics['test']['pred_std']:.4f}  "
              f"(sanity: minret_5d_pct is a percentage, so std should "
              f"clearly exceed 0.01 -- treat near-zero as constant output)")

    return {
        "best_params": best_params,
        "used_l1":     use_l1,
        "metrics":     all_metrics,
        "best_epoch":  result["best_epoch"],
        "total_time":  result["total_time"],
    }


# %% [markdown]
# ## Architecture Summary (from live model counts)

# %%
print("=" * 70)
print("  SPARSE MLP ARCHITECTURE SUMMARY")
print("=" * 70 + "\n")
for ds, info in architecture_info.items():
    print(f"  {ds}:")
    print(f"    Layer widths:      [{info['n_features']}, {info['n_subthemes']}, "
          f"{info['n_themes']}, 1]")
    print(f"    Active edges:      {info['active_edges']:,}")
    print(f"    Active parameters: {info['active_parameters']:,}")
    print()


# %% [markdown]
# ## Run All Experiments (3 Seeds × 16 Configurations)

# %%
configs    = len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)  # 16
total_runs = len(SEEDS) * configs                                   # 48

print("=" * 70)
print(f"  SPARSE MLP: 3 Seeds × 4 Splits × 2 Datasets × 2 Targets = {total_runs} runs")
print(f"  Seeds: {SEEDS}")
print(f"  Activation: SiLU (per-node)")
print(f"  Loss: binary=BCEWithLogitsLoss, "
      f"continuous=HuberLoss(delta=per-split, see huber_delta.json)")
print(f"  Early stop: binary=AUC, continuous=R²  (both maximize)")
print(f"  Optuna Phase 1: {N_TRIALS_NO_L1} trials (no L1), seeded TPE sampler")
print(f"  Optuna Phase 2: {N_TRIALS} trials (with L1 [1e-7, 1e-3]), seeded TPE sampler")
print(f"  Results saving to: {RESULTS_DIR}/seed_*/")
print("=" * 70)

all_results       = []
best_params_store = {}
completed         = 0
failed            = 0
total_start       = time.time()

for seed in SEEDS:

    # ── Set all random seeds for this pipeline run ──
    set_seed(seed)
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"

    print(f"\n\n{'═'*70}")
    print(f"  SEED {seed} — saving to {seed_results_dir}")
    print(f"{'═'*70}")

    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                try:
                    exp = run_single_experiment(
                        split_name, dataset, target_type, device,
                        seed=seed,
                        seed_results_dir=seed_results_dir,
                    )

                    all_results.append({
                        "seed":    seed,
                        "dataset": dataset,
                        "split":   split_name,
                        "target":  target_type,
                        "used_l1": exp["used_l1"],
                        **{f"test_{k}": v for k, v in exp["metrics"]["test"].items()
                           if not isinstance(v, np.ndarray)},
                        "best_epoch": exp["best_epoch"],
                        "time_s":     exp["total_time"],
                    })

                    key = (seed, dataset, target_type, split_name)
                    best_params_store[key] = {
                        **exp["best_params"], "used_l1": exp["used_l1"],
                    }
                    completed += 1

                    elapsed       = time.time() - total_start
                    rate          = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ Completed {completed}/{total_runs}  "
                          f"({elapsed/60:.0f}min elapsed, "
                          f"~{remaining_est/60:.0f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): seed={seed} "
                          f"{dataset}/{split_name}/{target_type}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"{'='*70}")


# %% [markdown]
# ## Cross-Seed Summary
#
# The key output: mean ± std across the 3 seeds for each configuration.
# This tells us how stable the full pipeline is to random initialisation.

# %%
if all_results:
    results_df = pd.DataFrame(all_results)

    # ── Save raw results CSV (all seeds, all configs) ──
    raw_path = RESULTS_DIR / "all_seeds_raw.csv"
    results_df.to_csv(raw_path, index=False)
    print(f"  Raw results saved to {raw_path}")

    # ═══════════════════════════════════════════════════════════════════════
    # BINARY AUC — per split (mean ± std across seeds)
    # ═══════════════════════════════════════════════════════════════════════
    binary_df = results_df[results_df["target"] == "binary"]

    print("\n" + "=" * 70)
    print("  SPARSE MLP — Binary Test AUC (mean ± std across 3 seeds)")
    print("=" * 70 + "\n")

    if "test_auc" in binary_df.columns and len(binary_df) > 0:
        agg = binary_df.groupby(["dataset", "split"])["test_auc"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = binary_df.groupby("dataset")["test_auc"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    # ═══════════════════════════════════════════════════════════════════════
    # CONTINUOUS R² — per split (mean ± std across seeds)
    # ═══════════════════════════════════════════════════════════════════════
    cont_df = results_df[results_df["target"] == "continuous"]

    print("\n" + "=" * 70)
    print("  SPARSE MLP — Continuous Test R² (mean ± std across 3 seeds)")
    print("=" * 70 + "\n")

    if "test_r2" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_r2"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_r2"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    # ═══════════════════════════════════════════════════════════════════════
    # CONTINUOUS DERIVED AUC — per split (mean ± std across seeds)
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  SPARSE MLP — Continuous Derived AUC (mean ± std across 3 seeds)")
    print("=" * 70 + "\n")

    if "test_derived_auc" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_derived_auc"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_derived_auc"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    # ═══════════════════════════════════════════════════════════════════════
    # CONTINUOUS MSE — per split (mean ± std across seeds)
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  SPARSE MLP — Continuous Test MSE (mean ± std across 3 seeds)")
    print("=" * 70 + "\n")

    if "test_mse" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_mse"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_mse"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())
        print(f"\n  NOTE: MSE is in minret_5d_pct units (percentage points "
              f"squared), NOT z-score units -- do not compare these MSE "
              f"values against any figure computed under the old pipeline.")

    # ═══════════════════════════════════════════════════════════════════════
    # PREDICTION STD SANITY CHECK
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  CONTINUOUS SANITY: Prediction std")
    print("  minret_5d_pct is a raw percentage; a healthy model's predictions")
    print("  should show meaningfully more than a trivial constant. Flag any")
    print("  run whose pred_std looks suspiciously close to zero relative to")
    print("  the others -- there is no longer a single fixed 0.01 threshold")
    print("  since this is no longer a pre-standardised z-score.")
    print("=" * 70 + "\n")

    if "test_pred_std" in cont_df.columns:
        for _, row in cont_df.iterrows():
            print(f"  seed={row['seed']}  {row['dataset']:<20} "
                  f"{row['split']:<10}  "
                  f"pred_std={row.get('test_pred_std', 0):.4f}")

    # ═══════════════════════════════════════════════════════════════════════
    # L1 USAGE SUMMARY
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  L1 USAGE: Did the Sparse MLP need L1?")
    print("=" * 70 + "\n")

    if "used_l1" in results_df.columns:
        n_l1  = results_df["used_l1"].sum()
        n_tot = len(results_df)
        print(f"  L1 selected: {n_l1}/{n_tot} runs across all seeds")
        for tt in TARGET_TYPES:
            subset = results_df[results_df["target"] == tt]
            n      = subset["used_l1"].sum()
            print(f"    {tt}: {n}/{len(subset)} runs used L1")
        print()
        if n_l1 == 0:
            print("  → Structural sparsity alone sufficient — L1 never needed")
        elif n_l1 == n_tot:
            print("  → L1 always helps — structural sparsity alone insufficient")
        else:
            print("  → Mixed — L1 helps for some configurations but not others")

    # ═══════════════════════════════════════════════════════════════════════
    # SEED STABILITY DIAGNOSTIC
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  SEED STABILITY: Best hyperparameters across seeds")
    print("=" * 70 + "\n")

    print(f"  {'Seed':>5} {'Dataset':<20} {'Target':<12} {'Split':<10} "
          f"{'lr':>10} {'wd':>10} {'bs':>5} {'reg_w':>10} {'L1?':>4}")
    print("  " + "-" * 90)
    for (seed, ds, tt, split), params in sorted(best_params_store.items()):
        rw   = params.get("reg_weight", 0)
        used = params.get("used_l1", False)
        print(f"  {seed:>5} {ds:<20} {tt:<12} {split:<10} "
              f"{params['lr']:>10.6f} {params['weight_decay']:>10.6f} "
              f"{params['batch_size']:>5} {rw:>10.2e} "
              f"{'yes' if used else 'no':>4}")

    # ═══════════════════════════════════════════════════════════════════════
    # SEED VARIANCE SUMMARY
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  SEED VARIANCE SUMMARY")
    print("=" * 70 + "\n")

    for tt in TARGET_TYPES:
        metric_col   = "test_auc"         if tt == "binary" else "test_derived_auc"
        metric_label = "AUC"              if tt == "binary" else "Derived AUC"
        subset = results_df[results_df["target"] == tt]

        if metric_col not in subset.columns or len(subset) == 0:
            continue

        print(f"  {tt.upper()} ({metric_label}):")
        per_config = subset.groupby(["dataset", "split"])[metric_col].agg(
            ["mean", "std"]
        )
        max_std  = per_config["std"].max()
        mean_std = per_config["std"].mean()
        print(f"    Mean seed std across configs: {mean_std:.4f}")
        print(f"    Max seed std across configs:  {max_std:.4f}")

        if max_std < 0.01:
            print(f"    → Very stable: seed choice barely matters")
        elif max_std < 0.03:
            print(f"    → Moderately stable: some sensitivity to initialisation")
        else:
            print(f"    → High variance: results depend substantially on seed")
        print()

    # ═══════════════════════════════════════════════════════════════════════
    # TIMING
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  TIMING")
    print("=" * 70 + "\n")

    for _, row in results_df.iterrows():
        print(f"  seed={row['seed']}  {row['dataset']:<20} "
              f"{row['split']:<10} {row['target']:<12} "
              f"best_epoch={row['best_epoch']:>3}  {row['time_s']:>6.1f}s")

    # ═══════════════════════════════════════════════════════════════════════
    # SAVE AGGREGATED SUMMARY CSV
    # ═══════════════════════════════════════════════════════════════════════
    summary_rows = []
    for tt in TARGET_TYPES:
        subset = results_df[results_df["target"] == tt]
        if len(subset) == 0:
            continue

        metric_cols = [c for c in subset.columns
                       if c.startswith("test_") and
                       subset[c].dtype in [np.float64, np.float32, float]]

        agg = subset.groupby(["dataset", "split"])[metric_cols].agg(
            ["mean", "std"]
        ).reset_index()

        agg.columns = [
            f"{c[0]}_{c[1]}" if c[1] else c[0]
            for c in agg.columns
        ]

        agg["target"] = tt
        summary_rows.append(agg)

    if summary_rows:
        summary_df   = pd.concat(summary_rows, ignore_index=True)
        summary_path = RESULTS_DIR / "cross_seed_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"\n  Cross-seed summary saved to {summary_path}")

else:
    print("\n  No results to display — all experiments failed.")


# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES (on Google Drive)")
print("=" * 70)

for seed in SEEDS:
    seed_dir = RESULTS_DIR / f"seed_{seed}"
    print(f"\n  ── seed_{seed}/ ──")
    for subdir in ["predictions", "metrics", "checkpoints", "optuna"]:
        d = seed_dir / subdir
        if d.exists():
            files = list(d.glob("sparse_mlp_*"))
            print(f"    {subdir}/: {len(files)} files")
        else:
            print(f"    {subdir}/: (not yet created)")

for fname in ["all_seeds_raw.csv", "cross_seed_summary.csv"]:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        print(f"\n  {fname}: ✓")
    else:
        print(f"\n  {fname}: (not yet created)")


# %% [markdown]
# ## Backtests (Seed-Averaged Signal)
#
# Predictions are averaged across the 3 seeds before backtesting.
# Averaging reduces noise and produces a single ensemble signal —
# this is both cleaner to interpret and what you would do in practice.
#
# One `run_full_backtest` call per dataset × split × target_type:
#   - Signal diagnostics (raw vs smoothed range)
#   - Simple timing: threshold chosen on val (net-of-cost Sortino)
#   - Simple timing cost sensitivity sweep
#   - Asymmetric risk-scaled: params found on val
#   - Risk-scaled cost sensitivity sweep
#   - Side-by-side comparison vs buy-and-hold
#
# Binary signal  : averaged y_prob  (go_cash_when="above")
# Continuous signal: averaged y_pred (go_cash_when="below")
#
# EDIT: results are now SAVED to disk, not just printed. See
# backtests/backtest_summary.csv and backtests/backtest_full_results.json.
#
# NOTE: evaluation.py's fixed continuous threshold grids
# (find_best_threshold / find_best_risk_scaled_params) still assume a
# roughly [-5, +2] z-score range from the old target. minret_5d_pct lives
# on a different scale, so these grids may sit partly or entirely outside
# where the model's actual predictions land, biasing the backtest toward
# a grid edge. This is a KNOWN, DEFERRED issue in evaluation.py -- not
# something to fix in this notebook. If backtest thresholds print a
# grid-edge warning here, that is this known issue surfacing, not a new bug.

# %%
def load_averaged_predictions(model_name, split_name, target_type, part,
                               seeds, results_dir):
    """
    Load predictions from each seed folder and average the signal.

    Binary:     averages y_prob across seeds
    Continuous: averages y_pred across seeds

    Returns (returns, avg_signal) as numpy arrays.
    Daily returns are identical across seeds (same data), so we take
    them from the first seed only.
    """
    signals = []
    returns = None

    for seed in seeds:
        seed_dir = results_dir / f"seed_{seed}"
        loaded = load_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            results_dir=seed_dir,
        )
        preds = loaded["predictions"]

        if returns is None:
            returns = preds["daily_return"].values

        if target_type == "binary":
            signals.append(preds["y_prob"].values)
        else:
            signals.append(preds["y_pred"].values)

    avg_signal = np.mean(np.stack(signals, axis=0), axis=0)
    return returns, avg_signal


def _make_json_safe(obj):
    """Recursively convert DataFrames/numpy types to plain JSON-serialisable types."""
    if isinstance(obj, dict):
        return {k: _make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame):
        return obj.reset_index().to_dict(orient="records")
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# %%
print("\n" + "=" * 70)
print("  SPARSE MLP — BACKTESTS (seed-averaged signal)")
print("  Signal = mean prediction across seeds 42, 123, 456")
print("=" * 70)

backtest_dir = RESULTS_DIR / "backtests"
backtest_dir.mkdir(parents=True, exist_ok=True)

backtest_rows    = []   # flat summary row per (dataset, split, target_type, strategy)
backtest_records = {}   # full nested dict, saved as one JSON for deep inspection

for dataset in DATASETS:
    for split_name in ALL_SPLITS:
        model_name = f"sparse_mlp_{dataset}"

        # ── Binary ──
        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "binary", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "binary", "test", SEEDS, RESULTS_DIR)

        bt_binary = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="above",
            model_name=f"{model_name} (binary)",
            split_name=split_name,
        )

        # ── Continuous ──
        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "continuous", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "continuous", "test", SEEDS, RESULTS_DIR)

        bt_continuous = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="below",
            model_name=f"{model_name} (continuous)",
            split_name=split_name,
        )

        # ── Store the full nested result for both target types ──
        key = f"{dataset}/{split_name}"
        backtest_records[key] = {
            "binary": bt_binary,
            "continuous": bt_continuous,
        }

        # ── Flat summary rows for a quick-scan CSV ──
        for target_type, bt in [("binary", bt_binary), ("continuous", bt_continuous)]:
            for strategy_name, strategy_key in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
                bt_result = bt[strategy_key]
                backtest_rows.append({
                    "dataset": dataset,
                    "split": split_name,
                    "target_type": target_type,
                    "strategy": strategy_name,
                    "sharpe": bt_result["sharpe"],
                    "sortino": bt_result["sortino"],
                    "annual_return": bt_result["annual_return"],
                    "max_drawdown": bt_result["max_drawdown"],
                    "cumulative_return": bt_result["cumulative_return"],
                    "avg_exposure": bt_result["avg_exposure"],
                    "annual_turnover": bt_result["annual_turnover"],
                    "buy_hold_sharpe": bt_result["buy_hold_sharpe"],
                    "buy_hold_sortino": bt_result["buy_hold_sortino"],
                    "buy_hold_cumulative": bt_result["buy_hold_cumulative"],
                })

# ── Save flat summary CSV (quick to scan/pivot) ──
backtest_summary_df = pd.DataFrame(backtest_rows)
backtest_summary_path = backtest_dir / "backtest_summary.csv"
backtest_summary_df.to_csv(backtest_summary_path, index=False)
print(f"\n  Backtest summary saved to {backtest_summary_path}")

# ── Save full nested results as JSON (includes cost-sensitivity tables, chosen params, etc.) ──
backtest_json_path = backtest_dir / "backtest_full_results.json"
with open(backtest_json_path, "w") as f:
    json.dump(_make_json_safe(backtest_records), f, indent=2, default=str)
print(f"  Full backtest results (incl. cost-sensitivity tables) saved to {backtest_json_path}")


# %% [markdown]
# ## Disconnect Runtime

# %%
print("All experiments complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()

Mounted at /content/drive
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
Loading taxonomies...
  agg_full_moments: 1699 features, 331 subthemes, 13 themes
  agg_means: 574 features, 128 subthemes, 13 themes
Device: Tesla T4 (CUDA)
  ✓ All masked parameters are exactly zero
  agg_full_moments: 2,043 active edges  (2,388 active params) — ✓ GPU forward pass OK
  ✓ All masked parameters are exactly zero
  agg_means: 715 active edges  (857 active params) — ✓ GPU forward pass OK

  GPU forward+backward (batch=256): 1.45ms
  Estimated per trial: 0.8s
  Estimated per seed (1120 trials): 0.2 hours
  Estimated total (3 seeds): 0.7 hours

  SPARSE MLP ARCHITECTURE SUMMARY

  agg_full_moments:
    Layer widths:      [1699, 331, 13, 1]
    Active edges:      2,043
    Active parameters: 2,388

  agg_means:
    Layer widths:      [574, 128, 13, 1]
    Active edges:      715
    Active parameters: 857

  SPARSE MLP: 3 Seeds × 4 Splits × 2 Datasets × 2 Targets = 48 runs
  Seeds: [42, 123, 456]
  Activ

  0%|          | 0/30 [00:00<?, ?it/s]

  no_L1 best AUC: 0.7649  params: {'lr': 0.0005887056496896425, 'weight_decay': 0.00019297863660804117, 'batch_size': 256}
  with_L1: 40 trials (0 already complete)


  0%|          | 0/40 [00:00<?, ?it/s]

  with_L1 best AUC: 0.8015  params: {'lr': 0.0012223059070087024, 'weight_decay': 0.0005305021656542928, 'batch_size': 64, 'reg_weight': 0.0009769656942728068}

  → With-L1 wins (seed=42, AUC=0.8015)
  Epoch    1 | Train loss 1.0962 | Val loss 0.7741  AUC 0.7544 | LR 1.2e-03
  Epoch   20 | Train loss 0.8365 | Val loss 0.4770  AUC 0.5258 | LR 3.1e-04
  Early stop at epoch 21. Best val AUC: 0.7544 at epoch 1
  Training complete in 4.3s

  Results (seed=42, reg_weight=9.77e-04):
    Train AUC: 0.7987
    Val AUC:   0.7544
    Test AUC:  0.7369
    Gap:       +0.0618

  ✓ Completed 1/48  (6min elapsed, ~290min remaining)

────────────────────────────────────────────────────────────
  seed=42 / agg_full_moments / Split_B / binary
────────────────────────────────────────────────────────────
  no_L1: 30 trials (0 already complete)


...

  with_L1 best R²: 0.3384  params: {'lr': 0.00023150895434387024, 'weight_decay': 5.575181094760599e-06, 'batch_size': 128, 'reg_weight': 6.686392360859417e-06}

  → No-L1 wins (seed=456, R²=0.3814)
  Epoch    1 | Train Huber 0.8202 | Val Huber 0.9618  MSE 2.9346  R² -0.0393 | LR 6.1e-03
  Epoch   20 | Train Huber 0.2580 | Val Huber 0.8518  MSE 2.2360  R² 0.2081 | LR 3.1e-03
  Early stop at epoch 27. Best val R²: 0.2293 at epoch 7
  Training complete in 2.4s

  Results (seed=456, no L1, huber_delta=2.4230):
    Train R²:    0.5245  (MSE=0.6814)
    Val R²:      0.2293  (MSE=2.1763)
    Test R²:     -0.1275  (MSE=1.1432)
    Derived AUC: 0.7528
    Pred std:    0.3566  (sanity: minret_5d_pct is a percentage, so std should clearly exceed 0.01 -- treat near-zero as constant output)

  ✓ Completed 47/48  (317min elapsed, ~7min remaining)

────────────────────────────────────────────────────────────
  seed=456 / agg_means / Split_D / continuous
──────────────────────────────────────────────

  0%|          | 0/30 [00:00<?, ?it/s]

  no_L1 best R²: 0.2124  params: {'lr': 0.0015435654026083772, 'weight_decay': 1.4808753737868237e-06, 'batch_size': 256}
  with_L1: 40 trials (0 already complete)


  0%|          | 0/40 [00:00<?, ?it/s]

  with_L1 best R²: 0.2548  params: {'lr': 0.004152399774307522, 'weight_decay': 3.340398650390741e-05, 'batch_size': 256, 'reg_weight': 0.00029043600718698444}

  → With-L1 wins (seed=456, R²=0.2548)
  Epoch    1 | Train Huber 1.5156 | Val Huber 1.6634  MSE 3.4618  R² -2.0806 | LR 4.2e-03
  Epoch   20 | Train Huber 0.4055 | Val Huber 0.5990  MSE 1.1947  R² -0.0632 | LR 4.2e-03
  Early stop at epoch 38. Best val R²: -0.0110 at epoch 18
  Training complete in 2.7s

  Results (seed=456, reg_weight=2.90e-04, huber_delta=2.4998):
    Train R²:    0.4962  (MSE=0.8603)
    Val R²:      -0.0110  (MSE=1.1361)
    Test R²:     -0.0798  (MSE=0.5351)
    Derived AUC: 0.6031
    Pred std:    0.3135  (sanity: minret_5d_pct is a percentage, so std should clearly exceed 0.01 -- treat near-zero as constant output)

  ✓ Completed 48/48  (324min elapsed, ~0min remaining)


  FINISHED: 48/48 completed, 0 failed
  Total time: 323.5 minutes (5.4 hours)
  Raw results saved to /content/drive/MyDrive/Thesis/Da